In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install requests pandas pyarrow lxml beautifulsoup4 pymupdf -q
# import fulltext_harvester_v2

In [ ]:
#!/usr/bin/env python3
"""
fulltext_preflight.py  —  Google Colab
======================================
Run this BEFORE re-running the full-text harvester on your FINAL included corpus.

WHAT IT DOES
  1. AUDIT — where you actually stand on the included studies only: how many have
     full text, how many have the target back-matter sections (data availability /
     ethics / author contributions), broken down by source and by bloc. Costs
     seconds, tells you whether a re-run is worth it and what to fix first.
  2. REWRITE THE MANIFEST so the harvester does the right thing on re-run:
       keep   -> successes WITH target sections        (finished; skipped)
       mark   -> successes WITHOUT target sections     (reprocess; upgrade only)
       drop   -> failures and rows outside the corpus  (retried fresh)
     Without this the harvester SKIPS failed rows (they are in the manifest), i.e.
     exactly the studies you want to retry.
  3. Optionally quarantines section/raw files for studies no longer in the corpus
     (moved, never deleted).

SAFE: writes a timestamped backup of the manifest first. Nothing is deleted.

SETUP
  !pip install pandas pyarrow -q
"""

import json, shutil, sys, time
from pathlib import Path
import pandas as pd

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
INCLUDED_FILE = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/merged_human_class2.xlsx"  # FINAL corpus
INCLUDE_COL   = "last_final_include"
INCLUDE_VAL   = "include"

FT_DIR   = "/content/drive/MyDrive/COGNOSX/ticks/fulltext"
SEC_DIR  = FT_DIR + "/sections"
RAW_DIR  = FT_DIR + "/raw"
MANIFEST = FT_DIR + "/fulltext_manifest.parquet"
AUDIT_CSV = FT_DIR + "/preflight_audit.csv"

REWRITE_MANIFEST = True     # False = audit only (dry run)
QUARANTINE_OUTSIDE = True  # True = move files of non-included studies aside

TARGET_SECTIONS = {"data_availability", "ethics", "author_contrib",
                   "code_availability", "funding", "conflicts"}
# sources considered STRUCTURED (tagged JATS/TEI) vs weak (regex on text)
STRUCTURED_HINTS = ("_xml", "grobid")


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def short_id(x): return S(x).rsplit("/", 1)[-1]
def dominant_bloc(sb):
    s = S(sb).lower()
    if "ssa" in s: return "ssa"
    if "maghreb" in s: return "maghreb"
    return "other"

def load_included():
    p = str(INCLUDED_FILE).lower()
    df = pd.read_excel(INCLUDED_FILE, dtype=str) if p.endswith((".xlsx", ".xls")) \
         else pd.read_parquet(INCLUDED_FILE)
    if INCLUDE_COL in df.columns:
        df = df[df[INCLUDE_COL].astype(str).str.strip().str.lower() == INCLUDE_VAL]
    df["oa_id"] = df["oa_id"].map(short_id)
    print(f"[in] included corpus: {len(df):,} studies", file=sys.stderr)
    return df

# =========================================================================== #
# 1. inspect what is actually on disk for the included corpus                 #
# =========================================================================== #
def inspect(df):
    ids = [short_id(x) for x in df["oa_id"]]
    bloc = (dict(zip(ids, df["study_bloc"].map(dominant_bloc)))
            if "study_bloc" in df.columns else {})
    rows = []
    for oid in ids:
        p = Path(SEC_DIR) / f"{oid}.json"
        rec = {"oa_id": oid, "bloc": bloc.get(oid, "other"),
               "has_sections_file": p.exists(), "source": "", "n_sections": 0,
               "has_targets": False, "structured": False, "section_types": ""}
        if p.exists():
            try:
                doc = json.loads(p.read_text(encoding="utf-8"))
                types = {s.get("type", "") for s in doc.get("sections", [])}
                rec.update({"source": doc.get("source", ""),
                            "n_sections": len(doc.get("sections", [])),
                            "has_targets": bool(types & TARGET_SECTIONS),
                            "structured": any(h in S(doc.get("source"))
                                              for h in STRUCTURED_HINTS),
                            "section_types": ";".join(sorted(t for t in types if t))})
            except Exception:
                rec["source"] = "unreadable"
    return pd.DataFrame(rows if rows else
                        [{"oa_id": o, "bloc": bloc.get(o, "other"),
                          "has_sections_file": False, "source": "", "n_sections": 0,
                          "has_targets": False, "structured": False,
                          "section_types": ""} for o in ids])


def inspect_fast(df):
    """Same as inspect() but avoids re-reading every JSON when possible."""
    ids = [short_id(x) for x in df["oa_id"]]
    bloc = (dict(zip(ids, df["study_bloc"].map(dominant_bloc)))
            if "study_bloc" in df.columns else {})
    have = {p.stem for p in Path(SEC_DIR).glob("*.json")} if Path(SEC_DIR).exists() else set()
    rows = []
    for oid in ids:
        r = {"oa_id": oid, "bloc": bloc.get(oid, "other"),
             "has_sections_file": oid in have, "source": "", "n_sections": 0,
             "has_targets": False, "structured": False, "section_types": ""}
        if oid in have:
            try:
                doc = json.loads((Path(SEC_DIR) / f"{oid}.json").read_text(encoding="utf-8"))
                types = {s.get("type", "") for s in doc.get("sections", [])}
                src = S(doc.get("source"))
                r.update({"source": src, "n_sections": len(doc.get("sections", [])),
                          "has_targets": bool(types & TARGET_SECTIONS),
                          "structured": any(h in src for h in STRUCTURED_HINTS),
                          "section_types": ";".join(sorted(t for t in types if t))})
            except Exception:
                r["source"] = "unreadable"
        rows.append(r)
    return pd.DataFrame(rows)


# =========================================================================== #
# 2. audit report                                                             #
# =========================================================================== #
def audit(st, man):
    n = len(st)
    have = int(st["has_sections_file"].sum())
    tgt = int(st["has_targets"].sum())
    stru = int(st["structured"].sum())
    print(f"\n===== PRE-FLIGHT AUDIT (included corpus, n={n:,}) =====")
    print(f"  full text on disk      : {have:>8,}  ({have/n:6.1%})")
    print(f"  ... structured (JATS/TEI): {stru:>8,}  ({stru/n:6.1%})")
    print(f"  ... with target sections : {tgt:>8,}  ({tgt/n:6.1%})")
    print(f"  MISSING full text      : {n-have:>8,}  ({(n-have)/n:6.1%})  <- to download")

    if have:
        print("\n  by source:")
        for k, v in st.loc[st["has_sections_file"], "source"].value_counts().items():
            print(f"    {k:<32}{v:>8,}")

    if st["bloc"].nunique() > 1:
        print("\n  coverage by bloc  (availability is itself an equity signal):")
        g = st.groupby("bloc").agg(n=("oa_id", "size"),
                                   have=("has_sections_file", "mean"),
                                   targets=("has_targets", "mean"),
                                   structured=("structured", "mean"))
        for b, r in g.iterrows():
            print(f"    {b:<10}n={int(r['n']):>7,}  full text {r['have']:6.1%}  "
                  f"structured {r['structured']:6.1%}  targets {r['targets']:6.1%}")

    # why the missing ones failed, from the old manifest
    if man is not None and len(man):
        miss = set(st.loc[~st["has_sections_file"], "oa_id"])
        fail = man[man["oa_id"].isin(miss) & ~man["ok"].fillna(False)]
        if len(fail) and "attempts" in fail.columns:
            print(f"\n  failure chains for missing studies (top 8 of {len(fail):,}):")
            for k, v in fail["attempts"].value_counts().head(8).items():
                print(f"    {v:>7,}  {k}")
            n_grobid = int(fail["attempts"].astype(str).str.contains("grobid_off").sum())
            if n_grobid:
                print(f"\n  [!] {n_grobid:,} of these were blocked by `grobid_off` — that "
                      f"guard returns BEFORE Unpaywall is tried.\n      Remove it (and/or "
                      f"set GROBID_URL) to give them a real attempt.")
    st.to_csv(AUDIT_CSV, index=False)
    print(f"\n[write] {AUDIT_CSV}", file=sys.stderr)


# =========================================================================== #
# 3. rewrite the manifest for a correct re-run                                #
# =========================================================================== #
def rewrite_manifest(st, man):
    ids = set(st["oa_id"])
    ts = time.strftime("%Y%m%d-%H%M%S")
    if man is not None and len(man):
        bak = MANIFEST.replace(".parquet", f".backup-{ts}.parquet")
        man.to_parquet(bak, index=False)
        print(f"[backup] {bak}", file=sys.stderr)

    # build a manifest from what is ACTUALLY on disk, scoped to the corpus
    keep = st[st["has_sections_file"] & st["has_targets"]].copy()
    upg  = st[st["has_sections_file"] & ~st["has_targets"]].copy()

    rows = []
    for _, r in keep.iterrows():
        rows.append({"oa_id": r["oa_id"], "source": r["source"], "ok": True,
                     "n_sections": int(r["n_sections"]), "n_refs": 0,
                     "has_targets": True, "attempts": "preflight_keep"})
    for _, r in upg.iterrows():
        rows.append({"oa_id": r["oa_id"], "source": r["source"], "ok": True,
                     "n_sections": int(r["n_sections"]), "n_refs": 0,
                     "has_targets": False, "attempts": "preflight_upgrade"})
    # failures and out-of-corpus rows are simply ABSENT -> harvester retries them
    new = pd.DataFrame(rows)
    new.to_parquet(MANIFEST, index=False)
    print(f"\n[manifest] rewritten: {len(keep):,} finished (skip), "
          f"{len(upg):,} marked for upgrade, "
          f"{len(st) - len(keep) - len(upg):,} will be retried fresh", file=sys.stderr)
    print(f"[write] {MANIFEST}", file=sys.stderr)


def quarantine_outside(st):
    ids = set(st["oa_id"])
    qdir = Path(FT_DIR) / "_not_in_corpus"
    (qdir / "sections").mkdir(parents=True, exist_ok=True)
    (qdir / "raw").mkdir(parents=True, exist_ok=True)
    moved = 0
    for p in Path(SEC_DIR).glob("*.json"):
        if p.stem not in ids:
            shutil.move(str(p), str(qdir / "sections" / p.name)); moved += 1
    for p in Path(RAW_DIR).glob("*.*"):
        if p.stem not in ids:
            shutil.move(str(p), str(qdir / "raw" / p.name))
    print(f"[quarantine] moved {moved:,} section files of non-included studies "
          f"to {qdir}", file=sys.stderr)


# =========================================================================== #
if __name__ == "__main__":
    df = load_included()

    # df_all = pd.read_excel(INCLUDED_PARQUET)
    # df = df_all[df_all["last_final_include"] =='include']

    print(f"[in] included corpus: {len(df):,} studies", file=sys.stderr)

    man = None
    if Path(MANIFEST).exists():
        man = pd.read_parquet(MANIFEST)
        man["oa_id"] = man["oa_id"].map(short_id)
        outside = (~man["oa_id"].isin({short_id(x) for x in df["oa_id"]})).sum()
        print(f"[manifest] {len(man):,} rows ({outside:,} not in the included corpus)",
              file=sys.stderr)

    st = inspect_fast(df)
    audit(st, man)

    if REWRITE_MANIFEST:
        rewrite_manifest(st, man)
    else:
        print("\n[dry run] REWRITE_MANIFEST=False — nothing changed", file=sys.stderr)

    if QUARANTINE_OUTSIDE:
        quarantine_outside(st)

    print("\n===== NEXT =====")
    print("1. In the harvester: REMOVE the `if not GROBID_URL: return None, \"grobid_off\"`")
    print("   early return  (it blocks Unpaywall entirely).")
    print("2. Set NCBI_KEY (free) -> PMC rate limit 3 -> 10 rps.")
    print("3. Ideally start GROBID and set GROBID_URL, so PDFs give structured TEI")
    print("   instead of regex-split text (better ethics / data-availability capture).")
    print("4. Point the harvester's IN_PARQUET at the included corpus and run.")
    print("   It will skip the finished, upgrade the weak, and retry the missing.")

In [ ]:
vv

In [ ]:
import shutil
shutil.copy(
  "/content/drive/MyDrive/COGNOSX/ticks/fulltext/fulltext_manifest.backup-20260912-163448.parquet",
  "/content/drive/MyDrive/COGNOSX/ticks/fulltext/fulltext_manifest.parquet")

In [ ]:
from pathlib import Path
print("SEC_DIR =", SEC_DIR)
print("exists:", Path(SEC_DIR).exists())
files = list(Path(SEC_DIR).glob("*.json"))
print("json files found:", len(files))
print("first 5:", [p.name for p in files[:5]])

# and check the ID format on both sides
import pandas as pd
#df = pd.read_parquet(INCLUDED_PARQUET)

df_all = pd.read_excel(INCLUDED_FILE)
df = df_all[df_all["last_final_include"] =='include']
print("corpus oa_id sample:", df["oa_id"].head(3).tolist())
print("file stem sample:", [p.stem for p in files[:3]])

In [ ]:
have = {p.stem for p in Path(SEC_DIR).glob("*.json")}
ids  = set(df["oa_id"])          # df = votre corpus filtré
print("corpus :", len(ids))
print("déjà téléchargés :", len(ids & have))
print("manquants :", len(ids - have))

In [ ]:
#!/usr/bin/env python3
"""
explore_corpus_and_fulltext.py  —  Google Colab
===============================================
Diagnostic / exploration BEFORE any download. Answers:
  1. Does the oa_id key actually match between the Excel corpus and the files?
  2. What is in the Excel (columns, include-column values, how many included)?
  3. What is in fulltext/sections and fulltext/raw (counts, extensions, sizes)?
  4. What is the real overlap -> how many are truly missing?
  5. What do the existing section files contain (which section types, sources)?

Read-only: downloads nothing, changes nothing.

SETUP
  !pip install pandas pyarrow openpyxl -q
"""

import json, random, sys
from collections import Counter
from pathlib import Path
import pandas as pd

# =========================================================================== #
# CONFIG                                                                        #
# =========================================================================== #
INCLUDED_FILE = "/content/drive/MyDrive/COGNOSX/ticks/preprocessing_data/merged_human_class2.xlsx"
INCLUDE_COL   = "last_final_include"     # set to "" to skip filtering
INCLUDE_VAL   = "include"
ID_COL        = "oa_id"

FT_DIR  = "/content/drive/MyDrive/COGNOSX/ticks/fulltext2"
SEC_DIR = FT_DIR + "/sections"
RAW_DIR = FT_DIR + "/raw"

SAMPLE_JSON = 300        # how many section files to open for content stats
TARGET_SECTIONS = {"data_availability", "ethics", "author_contrib",
                   "code_availability", "funding", "conflicts"}


def S(x):
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA else str(x)
def short_id(x): return S(x).rsplit("/", 1)[-1].strip()


# =========================================================================== #
# 1. explore the Excel / corpus file                                          #
# =========================================================================== #
def explore_corpus():
    p = str(INCLUDED_FILE).lower()
    df = (pd.read_excel(INCLUDED_FILE, dtype=str) if p.endswith((".xlsx", ".xls"))
          else pd.read_parquet(INCLUDED_FILE))
    print("=" * 70)
    print(f"CORPUS FILE: {Path(INCLUDED_FILE).name}")
    print("=" * 70)
    print(f"rows: {len(df):,}   columns: {len(df.columns)}")
    print("\ncolumns:")
    for c in df.columns:
        print(f"   - {c}")

    if INCLUDE_COL and INCLUDE_COL in df.columns:
        print(f"\nvalues in '{INCLUDE_COL}':")
        vc = df[INCLUDE_COL].fillna("(empty)").astype(str).str.strip().value_counts()
        for k, v in vc.head(15).items():
            print(f"   {k!r:<28}{v:>8,}")
        inc = df[df[INCLUDE_COL].astype(str).str.strip().str.lower() == INCLUDE_VAL.lower()]
        print(f"\n-> filtered on {INCLUDE_COL} == '{INCLUDE_VAL}': {len(inc):,} studies")
    else:
        if INCLUDE_COL:
            print(f"\n[!] column '{INCLUDE_COL}' NOT FOUND — using all rows")
        inc = df

    if ID_COL not in inc.columns:
        raise SystemExit(f"[!] id column '{ID_COL}' not in file")
    inc = inc.copy()
    inc["_sid"] = inc[ID_COL].map(short_id)
    bad = inc["_sid"].eq("").sum()
    dup = inc["_sid"].duplicated().sum()
    print(f"\nid column '{ID_COL}': {inc['_sid'].nunique():,} unique | "
          f"{bad:,} empty | {dup:,} duplicates")
    print(f"format sample: {inc['_sid'].head(5).tolist()}")
    return inc


# =========================================================================== #
# 2. explore the fulltext folders                                             #
# =========================================================================== #
def explore_folders():
    print("\n" + "=" * 70)
    print("FULLTEXT FOLDERS")
    print("=" * 70)
    out = {}
    for name, d in (("sections", SEC_DIR), ("raw", RAW_DIR)):
        p = Path(d)
        if not p.exists():
            print(f"\n{name}: DOES NOT EXIST ({d})"); out[name] = set(); continue
        files = list(p.iterdir())
        ext = Counter(f.suffix.lower() for f in files if f.is_file())
        size = sum(f.stat().st_size for f in files if f.is_file())
        print(f"\n{name}:  {len([f for f in files if f.is_file()]):,} files, "
              f"{size/1e9:.2f} GB")
        for e, n in ext.most_common():
            print(f"   {e or '(none)':<10}{n:>8,}")
        out[name] = {f.stem for f in files if f.is_file()}
        print(f"   stem sample: {list(out[name])[:5]}")
    return out


# =========================================================================== #
# 3. the overlap — the actual answer                                          #
# =========================================================================== #
def overlap(inc, folders):
    ids = set(inc["_sid"])
    sec = folders.get("sections", set())
    raw = folders.get("raw", set())
    print("\n" + "=" * 70)
    print("OVERLAP  (does the key match?)")
    print("=" * 70)
    both = ids & sec
    print(f"corpus studies              : {len(ids):,}")
    print(f"section files on disk       : {len(sec):,}")
    print(f"-> ALREADY have full text   : {len(both):,}  ({len(both)/max(len(ids),1):.1%})")
    print(f"-> MISSING, to download     : {len(ids - sec):,}")
    print(f"section files NOT in corpus : {len(sec - ids):,}  (from the old wider corpus)")
    print(f"raw files matching corpus   : {len(ids & raw):,}")

    if not both:
        print("\n[!] ZERO overlap — the keys do NOT match. Compare these:")
        print(f"    corpus : {sorted(ids)[:5]}")
        print(f"    files  : {sorted(sec)[:5]}")
    else:
        print("\n[ok] keys match — the earlier 0% audit was reading a different "
              "corpus file, not a key problem.")
    return both, ids - sec


# =========================================================================== #
# 4. what is actually inside the existing section files                       #
# =========================================================================== #
def inspect_sections(have_ids):
    if not have_ids:
        return
    print("\n" + "=" * 70)
    print(f"CONTENT OF EXISTING SECTION FILES (sample of {SAMPLE_JSON})")
    print("=" * 70)
    sample = random.Random(42).sample(sorted(have_ids), min(SAMPLE_JSON, len(have_ids)))
    src = Counter(); types = Counter(); n_tgt = 0; n_ok = 0; lang = Counter()
    for oid in sample:
        try:
            doc = json.loads((Path(SEC_DIR) / f"{oid}.json").read_text(encoding="utf-8"))
        except Exception:
            src["unreadable"] += 1; continue
        n_ok += 1
        src[doc.get("source", "?")] += 1
        lang[doc.get("language", "") or "(none)"] += 1
        t = {s.get("type", "") for s in doc.get("sections", [])}
        types.update(t)
        if t & TARGET_SECTIONS:
            n_tgt += 1
    print(f"readable: {n_ok}/{len(sample)}")
    print(f"\nwith >=1 TARGET section (data availability / ethics / contributions): "
          f"{n_tgt} ({n_tgt/max(n_ok,1):.1%})")
    print("\nby source:")
    for k, v in src.most_common():
        print(f"   {k:<32}{v:>6}")
    print("\nmost common section types:")
    for k, v in types.most_common(18):
        mark = " <-- TARGET" if k in TARGET_SECTIONS else ""
        print(f"   {k:<26}{v:>6}{mark}")
    print("\nlanguages:")
    for k, v in lang.most_common(8):
        print(f"   {k:<10}{v:>6}")


# =========================================================================== #
if __name__ == "__main__":
    inc = explore_corpus()
    folders = explore_folders()
    have, missing = overlap(inc, folders)
    inspect_sections(have)

    print("\n" + "=" * 70)
    print("SUMMARY")
    print("=" * 70)
    print(f"  already downloaded : {len(have):,}")
    print(f"  still to download  : {len(missing):,}")
    if missing:
        pd.DataFrame({"oa_id": sorted(missing)}).to_csv(
            FT_DIR + "/missing_ids.csv", index=False)
        print(f"  -> wrote {FT_DIR}/missing_ids.csv")